# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = ConcreteModel()

## 🔹 Sets

In [ ]:
model.TACHES = Set(initialize=['marche', 'cuisine', 'vaisselle', 'buanderie'])
model.PERSONNES = Set(initialize=['Eve', 'Steven'])
model.ARC = Set(dimen=2, initialize=[(i0,i1) for i0 in model.PERSONNES for i1 in model.TACHES])

## 🔹 Parameters

In [ ]:
model.Temps = Param(model.PERSONNES, model.TACHES, initialize={('Eve', 'marche'): 4.5, ('Eve', 'cuisine'): 7.8, ('Eve', 'vaisselle'): 3.6, ('Eve', 'buanderie'): 2.9, ('Steven', 'marche'): 4.9, ('Steven', 'cuisine'): 7.2, ('Steven', 'vaisselle'): 4.3, ('Steven', 'buanderie'): 3.1}, within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.X = Var(model.PERSONNES, model.TACHES, domain=Binary)

## 🔹 Constraints

In [ ]:
model.c_for_0 = ConstraintList()
for p in model.PERSONNES:
    model.c_for_0.add(sum(model.X[p, t] for t in model.TACHES) == 2)
model.c_for_1 = ConstraintList()
for t in model.TACHES:
    model.c_for_1.add(sum(model.X[p, t] for p in model.PERSONNES) == 1)
# @BIN/@GIN directive already handled in variable declarations

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(sum(model.Temps[p, t] * model.X[p, t] for t in model.TACHES) for p in model.PERSONNES), sense=minimize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

## 🎯 Valeur de la fonction objective

In [ ]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

## 📊 Valeurs optimales des variables

In [ ]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')